In [7]:
import pygame
import numpy as np
import math
import sys

# --- Configurações Iniciais ---
LARGURA_TELA, ALTURA_TELA = 800, 600
ALCANCE_MAXIMO = 300
ANGULOS_SENSOR = [-60, -30, 0, 30, 60] # Ângulos em graus relativos à frente do agente

BRANCO = (255, 255, 255)
PRETO = (0, 0, 0)
AZUL = (50, 150, 255)
VERMELHO = (255, 50, 50)
VERDE = (50, 255, 50)
CINZA = (150, 150, 150)
COR_TRAJETORIA = (255, 165, 0) # Laranja para destacar o caminho

def intersecao_linhas(p1, p2, p3, p4):
    """Calcula o ponto de interseção entre dois segmentos de reta, se existir."""
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    x4, y4 = p4

    denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if denom == 0:
        return None # Linhas paralelas ou coincidentes

    t = ((x1 - x3) * (y3 - y4) - (y1 - y3) * (x3 - x4)) / denom
    u = -((x1 - x2) * (y1 - y3) - (y1 - y2) * (x1 - x3)) / denom

    if 0 <= t <= 1 and 0 <= u <= 1:
        x_int = x1 + t * (x2 - x1)
        y_int = y1 + t * (y2 - y1)
        return (x_int, y_int)
    return None

def obter_segmentos_obstaculos(obstaculos):
    """Converte os retângulos de obstáculos e as bordas da tela em segmentos de reta."""
    segmentos = []
    # Bordas da janela (Paredes externas)
    segmentos.extend([
        ((0, 0), (LARGURA_TELA, 0)),
        ((LARGURA_TELA, 0), (LARGURA_TELA, ALTURA_TELA)),
        ((LARGURA_TELA, ALTURA_TELA), (0, ALTURA_TELA)),
        ((0, ALTURA_TELA), (0, 0))
    ])
    
    # Múltiplos retângulos
    for rect in obstaculos:
        p1 = rect.topleft
        p2 = rect.topright
        p3 = rect.bottomright
        p4 = rect.bottomleft
        segmentos.extend([(p1, p2), (p2, p3), (p3, p4), (p4, p1)])
        
    return segmentos

def main():
    pygame.init()
    tela = pygame.display.set_mode((LARGURA_TELA, ALTURA_TELA))
    pygame.display.set_caption("Módulo Sensorial - Raycasting com Trajetória")
    clock = pygame.time.Clock()
    fonte = pygame.font.SysFont("Arial", 14)

    # Estado do agente
    agente_x, agente_y = 400, 300
    agente_angulo = 0 # Em graus (0 aponta para a direita)
    velocidade = 4
    velocidade_rotacao = 3

    # Lista para armazenar a trajetória
    trajetoria = []
    MAX_PONTOS_TRAJETORIA = 1000

    # Obstáculos (Retângulos)
    obstaculos = [
        pygame.Rect(150, 100, 100, 50),
        pygame.Rect(550, 200, 80, 150),
        pygame.Rect(300, 400, 200, 40),
        pygame.Rect(100, 350, 60, 120)
    ]

    rodando = True
    while rodando:
        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                rodando = False

        # --- Controles do Agente ---
        teclas = pygame.key.get_pressed()
        if teclas[pygame.K_LEFT] or teclas[pygame.K_a]:
            agente_angulo -= velocidade_rotacao
        if teclas[pygame.K_RIGHT] or teclas[pygame.K_d]:
            agente_angulo += velocidade_rotacao
        
        rad_agente = math.radians(agente_angulo)
        moveu = False
        if teclas[pygame.K_UP] or teclas[pygame.K_w]:
            agente_x += math.cos(rad_agente) * velocidade
            agente_y += math.sin(rad_agente) * velocidade
            moveu = True
        if teclas[pygame.K_DOWN] or teclas[pygame.K_s]:
            agente_x -= math.cos(rad_agente) * velocidade
            agente_y -= math.sin(rad_agente) * velocidade
            moveu = True

        # Adiciona a posição atual na trajetória apenas se o robô andou ou se a lista está vazia
        if moveu or len(trajetoria) == 0:
            trajetoria.append((int(agente_x), int(agente_y)))
            if len(trajetoria) > MAX_PONTOS_TRAJETORIA:
                trajetoria.pop(0)

        # --- Renderização Inicial ---
        tela.fill(BRANCO)
        segmentos = obter_segmentos_obstaculos(obstaculos)

        # Renderiza os obstáculos
        for obs in obstaculos:
            pygame.draw.rect(tela, CINZA, obs)

        # Renderiza a trajetória
        if len(trajetoria) >= 2:
            pygame.draw.lines(tela, COR_TRAJETORIA, False, trajetoria, 2)

        # --- Lógica dos Sensores (Raycasting) ---
        ponto_origem = (agente_x, agente_y)

        for angulo_relativo in ANGULOS_SENSOR:
            angulo_absoluto = math.radians(agente_angulo + angulo_relativo)
            
            # Ponto final assumindo que não há colisões
            ponto_final_teorico = (
                agente_x + math.cos(angulo_absoluto) * ALCANCE_MAXIMO,
                agente_y + math.sin(angulo_absoluto) * ALCANCE_MAXIMO
            )

            menor_distancia = ALCANCE_MAXIMO
            ponto_colisao = ponto_final_teorico
            colidiu = False

            # Verifica colisão com cada segmento do ambiente
            for seg in segmentos:
                intersecao = intersecao_linhas(ponto_origem, ponto_final_teorico, seg[0], seg[1])
                if intersecao:
                    distancia = math.hypot(intersecao[0] - agente_x, intersecao[1] - agente_y)
                    if distancia < menor_distancia:
                        menor_distancia = distancia
                        ponto_colisao = intersecao
                        colidiu = True

            # Aplicação do ruído Gaussiano (média 0, desvio-padrão 2.0)
            ruido = np.random.normal(0, 2.0)
            leitura_sensor = menor_distancia + ruido
            
            # Limita o valor exibido para não ser menor que 0 ou maior que o alcance máximo devido ao ruído
            leitura_sensor = max(0.0, min(float(leitura_sensor), float(ALCANCE_MAXIMO)))

            # --- Renderização do Feixe ---
            cor_feixe = VERMELHO if colidiu else VERDE
            pygame.draw.line(tela, cor_feixe, ponto_origem, ponto_colisao, 2)
            pygame.draw.circle(tela, cor_feixe, (int(ponto_colisao[0]), int(ponto_colisao[1])), 4)

            # --- Renderização do Valor ---
            texto_valor = fonte.render(f"{leitura_sensor:.1f}", True, PRETO)
            
            # Posiciona o texto um pouco antes do final do feixe para facilitar a leitura
            pos_texto_x = agente_x + math.cos(angulo_absoluto) * (menor_distancia - 30)
            pos_texto_y = agente_y + math.sin(angulo_absoluto) * (menor_distancia - 30)
            
            # Se a distância for muito curta, desenha próximo à origem
            if menor_distancia < 40:
                pos_texto_x = ponto_colisao[0]
                pos_texto_y = ponto_colisao[1]

            tela.blit(texto_valor, (pos_texto_x, pos_texto_y))

        # Renderiza o agente
        pygame.draw.circle(tela, AZUL, (int(agente_x), int(agente_y)), 10)
        # Linha indicando a direção do agente
        direcao_x = agente_x + math.cos(rad_agente) * 20
        direcao_y = agente_y + math.sin(rad_agente) * 20
        pygame.draw.line(tela, PRETO, (agente_x, agente_y), (direcao_x, direcao_y), 3)

        pygame.display.flip()
        clock.tick(60)

    pygame.quit()
    sys.exit()

if __name__ == "__main__":
    main()

SystemExit: 